# DiffusionGemma-Jev (`djev`) on Google Colab Pro (`A100` / `L4`)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/taeold/djev-run/blob/main/colab.ipynb)

> **Colab Pro `A100` / `L4` Required by Default:** This notebook requests an **NVIDIA A100 (`40 GB` / `80 GB`)** or **NVIDIA L4 (`24 GB`)** (`gpuClass: premium`, `machine_shape: hm`) so all `17.53 GiB` of weights stay 100% in GPU VRAM (`~45 ms` on A100 HBM2e, `~65 ms` on L4). If Colab connects you to a default `T4`, click **`Runtime -> Change runtime type -> Hardware accelerator -> A100 GPU or L4 GPU`** (or top-right dropdown arrow next to `T4 -> Change runtime type`).

Run **DiffusionGemma-Jev** (`nvidia/diffusiongemma-26B-A4B-it-NVFP4`, `26B` total parameters, `4B` active per token across `128` experts) directly inside Google Colab Pro on an **NVIDIA A100 (`40 GB` / `80 GB`)** or **NVIDIA L4 (`24 GB`)** GPU, and play the built-in 1-step diffusion games (`/tetris`, `/dino`, and `/snake`) live inside the notebook.

| Colab Runtime | GPU / Compute Capability | Usable VRAM | `nvidia/diffusiongemma-26B-A4B-it-NVFP4` (`17.53 GiB`) | Active vLLM Kernel Path |
| :--- | :--- | :--- | :--- | :--- |
| **Colab Pro (`A100`)** | NVIDIA A100 (`SM 8.0`) | `40.0 GiB` / `80.0 GiB` | **Recommended (`~45 ms/step`, 100% VRAM)** (`KV_CACHE_GB=8`) | **Marlin `W4A16` NVFP4 MoE** (`MarlinExperts`) + `BF16` Attention |
| **Colab Pro (`L4`)** | NVIDIA L4 (`SM 8.9`) | `22.5 GiB` (`24 GB`) | **Supported (`~65 ms/step`, 100% VRAM)** (`KV_CACHE_GB=2`) | **Marlin `W4A16` NVFP4 MoE** (`MarlinExperts`) + `BF16` Attention |
| **Colab Free (`T4`)** | NVIDIA T4 (`SM 7.5`) | `15.0 GiB` (`16 GB`) | **Blocked by default** (`ALLOW_SLOW_T4_OFFLOAD = False`; requires `5 GB` PCIe CPU offload at `~400-600 ms/step`) | Switch to `A100` or `L4` in `Runtime -> Change runtime type` |

In [1]:
import os
import subprocess

ALLOW_SLOW_T4_OFFLOAD = False

try:
    smi_out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,compute_cap", "--format=csv,noheader,nounits"],
        text=True,
    ).strip().splitlines()[0]
    gpu_name, total_mib, free_mib, compute_cap = [x.strip() for x in smi_out.split(",")]
    vram_gib = float(total_mib) / 1024.0
    free_gib = float(free_mib) / 1024.0
    sm_major, sm_minor = [int(x) for x in compute_cap.split(".")]
except Exception as e:
    raise RuntimeError(
        "No GPU detected via nvidia-smi. In Colab, click Runtime -> Change runtime type -> "
        "Hardware accelerator -> select A100 GPU or L4 GPU."
    ) from e

if vram_gib < 22.0 and not ALLOW_SLOW_T4_OFFLOAD:
    raise RuntimeError(
        "Detected NVIDIA T4 (15 GB). T4 requires 5 GB PCIe CPU offload (~400-600 ms/step). "
        "Please switch to an A100 or L4 GPU via: Runtime -> Change runtime type "
        "-> Hardware accelerator -> A100 GPU or L4 GPU (or set ALLOW_SLOW_T4_OFFLOAD = True in this cell)."
    )

if vram_gib < 22.0:
    KV_CACHE_GB = 0.5
    CPU_OFFLOAD_GB = 5.0
    GPU_UTIL = 0.85
    DTYPE = "float16"
elif vram_gib < 30.0:
    KV_CACHE_GB = 1.5
    CPU_OFFLOAD_GB = 0.0
    GPU_UTIL = 0.90
    DTYPE = "auto"
elif vram_gib < 60.0:
    KV_CACHE_GB = 8.0
    CPU_OFFLOAD_GB = 0.0
    GPU_UTIL = 0.85
    DTYPE = "auto"
else:
    KV_CACHE_GB = 8.0
    CPU_OFFLOAD_GB = 0.0
    GPU_UTIL = 0.40
    DTYPE = "auto"

os.environ["KV_CACHE_GB"] = str(KV_CACHE_GB)
os.environ["CPU_OFFLOAD_GB"] = str(CPU_OFFLOAD_GB)
os.environ["GPU_UTIL"] = str(GPU_UTIL)
os.environ["DTYPE"] = DTYPE
os.environ["CANVAS"] = "256"
os.environ["DISABLE_MM"] = "1"

print(f"GPU Device         : {gpu_name} (SM {sm_major}.{sm_minor})")
print(f"Total / Free VRAM  : {vram_gib:.2f} GiB total / {free_gib:.2f} GiB free (0.00 GiB used by notebook kernel)")
print(f"Checkpoint Weights : 17.53 GiB (nvidia/diffusiongemma-26B-A4B-it-NVFP4)")
print(f"MoE Kernel Backend : vLLM Marlin W4A16 FP4 (dtype={DTYPE}, gpu_util={GPU_UTIL}, cpu_offload_gb={CPU_OFFLOAD_GB})")
print(f"Configured KV Cache: {KV_CACHE_GB} GiB (CANVAS=256, DISABLE_MM=1)")

GPU Device         : NVIDIA A100-SXM4-40GB (SM 8.0)
Total / Free VRAM  : 39.56 GiB total / 39.56 GiB free (0.00 GiB used by notebook kernel)
Checkpoint Weights : 17.53 GiB (nvidia/diffusiongemma-26B-A4B-it-NVFP4)
MoE Kernel Backend : vLLM Marlin W4A16 FP4 (dtype=auto, gpu_util=0.85, cpu_offload_gb=0.0)
Configured KV Cache: 8.0 GiB (CANVAS=256, DISABLE_MM=1)


## Step 1: Install `djev-run` & Start In-Process `nvidia/diffusiongemma-26B-A4B-it-NVFP4` Server

The cell below extracts the pre-built `vLLM` block-diffusion runtime (`PR #58216` constrained vocab projection + `PR #58226` fused Triton row-stats sampler) from `ghcr.io/taeold/djev-run:latest`, downloads `nvidia/diffusiongemma-26B-A4B-it-NVFP4` (`17.53 GiB`) into `/dev/shm/dgemma` (or `/content/dgemma` with a `/dev/shm/dgemma` symlink if `/dev/shm` has `< 22 GiB` free), and launches the in-process `djev` server on `http://127.0.0.1:8080`.

In [2]:
import glob
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request

DJEV_PORT = 8080
DJEV_BASE_URL = f"http://127.0.0.1:{DJEV_PORT}"

def install_and_start_djev():
    # 1. Check if server is already listening
    for p in (DJEV_PORT, 8898):
        try:
            r = json.loads(urllib.request.urlopen(f"http://127.0.0.1:{p}/health", timeout=2).read().decode())
            if r.get("status") == "ok":
                print(f"[djev] Connected to active in-process server at http://127.0.0.1:{p}")
                return f"http://127.0.0.1:{p}"
        except Exception:
            pass

    # 2. Ensure Python 3.12 is available for the pre-built manylinux cp312/abi3 runtime
    if not shutil.which("python3.12"):
        print("[djev] Installing standalone Python 3.12 via uv...")
        subprocess.run("curl -LsSf https://astral.sh/uv/install.sh | sh", shell=True, check=True)
        uv_bin = os.path.expanduser("~/.local/bin/uv")
        subprocess.run([uv_bin, "python", "install", "3.12"], check=True)
        py312_path = subprocess.check_output([uv_bin, "python", "find", "3.12"], text=True).strip()
        subprocess.run(["ln", "-sf", py312_path, "/usr/local/bin/python3.12"], check=True)
        uv_bin = os.path.expanduser("~/.local/bin/uv")
        if os.path.exists(uv_bin):
            subprocess.run([uv_bin, "pip", "install", "--python", "/usr/local/bin/python3.12", "-q", "pybase64"], check=False)

    # 3. Stream pre-compiled vLLM block-diffusion layers from ghcr.io/taeold/djev-run:latest
    if not os.path.exists("/opt/dgemma/structured_server.py"):
        print("[djev] Extracting pre-compiled vLLM block-diffusion runtime from ghcr.io/taeold/djev-run:latest...")
        tok = json.loads(urllib.request.urlopen("https://ghcr.io/token?scope=repository:taeold/djev-run:pull", timeout=15).read().decode())["token"]
        hdrs = {
            "Authorization": f"Bearer {tok}",
            "Accept": "application/vnd.oci.image.manifest.v1+json, application/vnd.docker.distribution.manifest.v2+json, application/vnd.oci.image.index.v1+json",
        }
        m = json.loads(urllib.request.urlopen(urllib.request.Request("https://ghcr.io/v2/taeold/djev-run/manifests/latest", headers=hdrs), timeout=15).read().decode())
        if "manifests" in m:
            dig = [x["digest"] for x in m["manifests"] if x.get("platform", {}).get("architecture") == "amd64"][0]
            m = json.loads(urllib.request.urlopen(urllib.request.Request(f"https://ghcr.io/v2/taeold/djev-run/manifests/{dig}", headers=hdrs), timeout=15).read().decode())
        cfg = json.loads(urllib.request.urlopen(urllib.request.Request(f"https://ghcr.io/v2/taeold/djev-run/blobs/{m['config']['digest']}", headers={"Authorization": f"Bearer {tok}"}), timeout=15).read().decode())

        total_layers = len(m["layers"])
        idx = 0
        for h in cfg.get("history", []):
            if not h.get("empty_layer", False):
                cmd = h.get("created_by", "")
                is_cuda_compat = idx == 2
                is_base_pkg = 21 <= idx <= 44 and any(k in cmd for k in ("uv pip install", "overlay_vllm.py")) and "kv_connectors" not in cmd
                is_latest_overlay = idx >= total_layers - 6 and any(k in cmd for k in ("patch_vllm.py", "COPY server/", "COPY entrypoint.sh"))
                if is_cuda_compat or is_base_pkg or is_latest_overlay:
                    blob = m["layers"][idx]["digest"]
                    sz_mb = m["layers"][idx]["size"] / (1024 ** 2)
                    print(f"  -> Extracting layer {idx} ({sz_mb:.1f} MiB)...", flush=True)
                    url = f"https://ghcr.io/v2/taeold/djev-run/blobs/{blob}"
                    blob_tok = json.loads(urllib.request.urlopen("https://ghcr.io/token?scope=repository:taeold/djev-run:pull", timeout=15).read().decode())["token"]
                    subprocess.run(
                        f'curl -fSL --retry 3 --retry-delay 2 -H "Authorization: Bearer {blob_tok}" "{url}" | tar -xz --warning=no-unknown-keyword --overwrite -C /',
                        shell=True,
                        check=True,
                    )
                idx += 1

        # Fetch latest entrypoint and game UIs (/tetris, /dino, /snake) from GitHub
        for fname in ("tetris.html", "dino.html", "snake.html", "ultra_fast_entrypoint.sh"):
            url = f"https://raw.githubusercontent.com/taeold/djev-run/main/{fname}"
            dest = "/entrypoint.sh" if fname == "ultra_fast_entrypoint.sh" else f"/opt/dgemma/{fname}"
            urllib.request.urlretrieve(url, dest)
        os.chmod("/entrypoint.sh", 0o755)

    # 4. Download nvidia/diffusiongemma-26B-A4B-it-NVFP4 (17.53 GiB) into /dev/shm/dgemma (or /content/dgemma if /dev/shm < 22 GiB)
    if not os.path.exists("/dev/shm/dgemma/.ready"):
        shm_free_gib = shutil.disk_usage("/dev/shm").free / (1024 ** 3)
        target_dir = "/dev/shm/dgemma" if shm_free_gib >= 22.0 else "/content/dgemma"
        os.makedirs(target_dir, exist_ok=True)
        if target_dir != "/dev/shm/dgemma":
            if os.path.islink("/dev/shm/dgemma") or os.path.exists("/dev/shm/dgemma"):
                shutil.rmtree("/dev/shm/dgemma", ignore_errors=True)
            os.symlink(target_dir, "/dev/shm/dgemma")
        print(f"[djev] Downloading nvidia/diffusiongemma-26B-A4B-it-NVFP4 (17.53 GiB) into {target_dir} (/dev/shm free: {shm_free_gib:.1f} GiB)...", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub[hf_transfer]"], check=True)
        os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id="nvidia/diffusiongemma-26B-A4B-it-NVFP4", local_dir=target_dir)
        open("/dev/shm/dgemma/.ready", "w").close()

    # 5. Configure LD_LIBRARY_PATH & PYTHONPATH and launch in-process server with live log streaming
    site_pkg = "/usr/local/lib/python3.12/dist-packages"
    nv_libs = [p for p in glob.glob(f"{site_pkg}/nvidia/*/lib") if os.path.isdir(p)]
    ld_paths = ["/usr/local/cuda-13.0/compat", "/usr/local/cuda-13.0/targets/x86_64-linux/lib", f"{site_pkg}/torch/lib"] + nv_libs
    if os.environ.get("LD_LIBRARY_PATH"):
        ld_paths.append(os.environ["LD_LIBRARY_PATH"])
    env = os.environ.copy()
    env["LD_LIBRARY_PATH"] = ":".join(ld_paths)
    env["PYTHONPATH"] = f"{site_pkg}:/opt/dgemma:" + env.get("PYTHONPATH", "")

    print("[djev] Launching in-process vLLM + System-1 server on port 8080...", flush=True)
    log_path = "/tmp/djev_server.log"
    log_file = open(log_path, "w")
    proc = subprocess.Popen(["/entrypoint.sh"], stdout=log_file, stderr=subprocess.STDOUT, env=env)
    log_reader = open(log_path, "r")
    t_start = time.time()
    while time.time() - t_start < 300:
        new_lines = log_reader.read()
        if new_lines:
            print(new_lines, end="", flush=True)
        if proc.poll() is not None:
            remaining = log_reader.read()
            if remaining:
                print(remaining, end="", flush=True)
            raise RuntimeError(f"Server process exited early with code {proc.returncode}. See log output above.")
        try:
            r = json.loads(urllib.request.urlopen(f"http://127.0.0.1:{DJEV_PORT}/health", timeout=2).read().decode())
            if r.get("status") == "ok":
                print(f"[djev] Server ready in {time.time() - t_start:.1f}s at http://127.0.0.1:{DJEV_PORT}", flush=True)
                return f"http://127.0.0.1:{DJEV_PORT}"
        except Exception:
            time.sleep(1.5)
    remaining = log_reader.read()
    if remaining:
        print(remaining, end="", flush=True)
    raise RuntimeError("Server timed out after 300s. See live log output above.")

DJEV_BASE_URL = install_and_start_djev()

[djev] Connected to active in-process server at http://127.0.0.1:8080


## Step 2: 1-Step `/v1/systemone` Evaluation Check (`choice`, `score`, `noul`)

Verify the local in-process engine with a single `POST /v1/systemone` request (`steps=1, samples=1`) that evaluates multi-class `choice` routing, a 1-to-4 `score` rubric, and a binary `noul` probability in one forward step.

In [3]:
import json
import time
import urllib.request

payload = {
    "model": "djev-dgemma",
    "steps": 1,
    "samples": 1,
    "state": {
        "ticket_id": "TCK-9042",
        "customer_tier": "enterprise",
        "text": (
            "Hi team, I was double-charged $149.00 on invoice INV-2026-8841 yesterday "
            "and my production API key is currently locked. Please refund the duplicate charge."
        ),
    },
    "questions": {
        "department": {
            "type": "choice",
            "instructions": "Which team should handle this ticket?",
            "criteria": {
                "billing": "Payments, invoices, duplicate charges, and refunds",
                "technical": "API errors, SDK bugs, and infrastructure outages",
                "sales": "Enterprise upgrades and contract renewals",
            },
        },
        "urgency": {
            "type": "score",
            "instructions": "Rate ticket urgency from 1 (low) to 4 (critical)",
            "criteria": [
                "General question, no service impact",
                "Minor issue with workaround",
                "Billing error or locked production access",
                "Complete multi-tenant outage",
            ],
        },
        "refund_requested": {
            "type": "noul",
            "instructions": "Is the customer explicitly requesting a refund?",
        },
    },
}

t0 = time.time()
req = urllib.request.Request(
    f"{DJEV_BASE_URL}/v1/systemone",
    data=json.dumps(payload).encode(),
    headers={"Content-Type": "application/json"},
)
resp = json.loads(urllib.request.urlopen(req, timeout=30).read().decode())
rtt_ms = round((time.time() - t0) * 1000, 1)
srv_ms = round(resp.get("diagnostics", {}).get("timing", {}).get("total_ms", rtt_ms), 1)
ans = resp["answers"]

print(f"1-Step System-1 Read Complete in {srv_ms} ms GPU ({rtt_ms} ms total)")
print("-" * 78)
print(f"{'SLOT ID':<18} | {'TYPE':<8} | {'PREDICTION':<22} | {'CONFIDENCE / SCORE'}")
print("-" * 78)
print(f"{'department':<18} | {'choice':<8} | {ans['department']['choice']:<22} | {ans['department']['confidence']*100:5.1f}% (probs: {json.dumps({k: round(v, 3) for k, v in ans['department']['probabilities'].items()})})")
print(f"{'urgency':<18} | {'score':<8} | {'level ' + str(round(ans['urgency']['score'])) + '/4':<22} | {ans['urgency']['score']:.2f} / 4.00 (conf: {ans['urgency']['confidence']*100:5.1f}%)")
print(f"{'refund_requested':<18} | {'noul':<8} | {str(ans['refund_requested']['noul'] >= 0.5):<22} | P(yes) = {ans['refund_requested']['noul']*100:5.1f}%")
print("-" * 78)

1-Step System-1 Read Complete in 65.9 ms GPU (142.0 ms total)
------------------------------------------------------------------------------
SLOT ID            | TYPE     | PREDICTION             | CONFIDENCE / SCORE
------------------------------------------------------------------------------
department         | choice   | billing                |  99.3% (probs: {"billing": 0.993, "technical": 0.007, "sales": 0.0})
urgency            | score    | level 2/4              | 2.00 / 4.00 (conf:  99.9%)
refund_requested   | noul     | True                   | P(yes) = 100.0%
------------------------------------------------------------------------------


## Step 3: Play `/tetris`, `/dino`, and `/snake` Live Inside Colab

Set `DEMO = "/tetris"`, `"/dino"`, or `"/snake"` and run the cell below to embed the live game UI served from port `8080` on your Colab GPU.

In [4]:
# Choose a built-in game: "/tetris", "/dino", or "/snake"
DEMO = "/tetris"

try:
    from google.colab import output
    print(f"Embedding {DEMO} from Colab GPU server (port 8080)...")
    output.serve_kernel_port_as_iframe(8080, path=DEMO, height=660)
except ImportError:
    print("Available demo routes on local server:")
    for r in ("/tetris", "/dino", "/snake"):
        print(f"  -> {DJEV_BASE_URL}{r}")

Embedding /tetris from Colab GPU server (port 8080)...
